# Assigned concepts vs billed movements

PUBLIC SYNTHETIC DEMONSTRATION — no production files, data or connection.

Compare line counts AND amounts. An equal grand total of nine lines still hides missing and extra billing at individual keys. Synthetic manual-adjustment flags provide bounded evidence; they do not reproduce private academic metadata.

Monetary detail uses integer cents. All identifiers, amounts, rules and dates are invented.

## 1. Load the offline contract

Run from the repository or the notebooks directory. No environment secrets are read.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import HTML, display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "data" / "synthetic").is_dir())
sys.path.insert(0, str(root / "src"))
from financial_controls.demo import load_inputs
from financial_controls.controls import enrollment_control, billing_control, scholarship_control
from financial_controls.reporting import render_html

period = "2030-09"  # explicit fictitious scope; never the machine date
data = load_inputs(root)

In [ ]:
print("assigned")
display(data["assigned"])

print("billed")
display(data["billed"])

## 2. Compare and retain the original difference

The shared engine validates keys and cardinality. Actual minus expected determines OK / DE MÁS / DE MENOS. Review status is separate.

In [ ]:
result = billing_control(data["assigned"], data["billed"], period=period)
display(result.detail)

## 3. Explain only what the evidence can justify

Each evidence ID is consumed once. A partial allocation leaves the case pending. Line and amount capacities are independent; this is aggregate explanation, not exact invoice matching.

In [ ]:
display(result.allocations)
display(result.detail.loc[result.detail['review_status'].eq('PENDIENTE')])
display(result.unassigned)

## 4. Reconstruct source totals

A bridge that reconciles can still have unresolved cases. Unassigned amounts and keyed net/absolute residuals are reported separately. The scholarship example intentionally has an unassigned amount.

In [ ]:
display(pd.DataFrame(result.kpis.items(), columns=['metric', 'value']))
assert result.kpis["expected_lines"] == result.kpis["actual_lines"] == 9
assert result.kpis["mother_difference_cents"] == -4000
assert result.kpis["pending_net_cents"] == -23000
assert result.kpis["pending_cases"] == 2
for measure in ["lines", "cents"]:
    assert result.kpis[f"mother_difference_{measure}"] == (
        result.kpis[f"detail_difference_{measure}"] + result.kpis[f"unassigned_{measure}"]
    )

## 5. Review output and limits

The CLI exports the same result as CSV/JSON/HTML. This notebook does not write to Oracle. See docs/business_rules.md for synthetic policy and docs/limitations.md for aggregation and evidence limits.

In [ ]:
display(HTML(render_html(result, "Assigned concepts vs billed movements | synthetic | " + period)))